# Exercise 1: Spark Fundamentals
- RDD Basics (tạo RDD, filter, map, tính count/sum/average)
- Text Processing (flatMap, reduceByKey, sortBy)
- Understanding Partitions (getNumPartitions, repartition, mapPartitions)

## Task 1.1: RDD Basics

Tạo SparkSession để kết nối tới cluster.

`SparkSession.builder` là entry point của mọi ứng dụng Spark:
- `.appName("...")` đặt tên hiển thị trên Spark Master UI
- `.master("spark://spark-master:7077")` địa chỉ Spark Master trong Docker
- `.getOrCreate()` tạo session mới hoặc dùng lại nếu đã có

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Exercise1") \
    .master("spark://spark-master:7077") \
    .getOrCreate()

sc = spark.sparkContext
print(f"Spark version: {spark.version}")
print(f"Master: {sc.master}")

### Tạo RDD từ range 1 đến 1000

`sc.parallelize(collection)` biến một Python list/range thành RDD phân tán trên cluster.

RDD (Resilient Distributed Dataset) là cấu trúc dữ liệu cốt lõi của Spark:
- **Immutable**: không thay đổi được, mỗi operation tạo RDD mới
- **Lazy**: transformation không chạy ngay, chỉ chạy khi gặp action
- **Distributed**: dữ liệu chia nhỏ trên nhiều worker node

In [ ]:
numbers = sc.parallelize(range(1, 1001))
print(f"Số phần tử: {numbers.count()}")
print(f"Số partition: {numbers.getNumPartitions()}")

### Filter số chia hết cho 7

`rdd.filter(func)` là **transformation** — trả về RDD mới chỉ chứa phần tử mà `func` trả về `True`.

Lưu ý: `filter` chưa thực thi ngay. Spark ghi nhớ DAG (Directed Acyclic Graph) và chỉ chạy khi gặp action như `count()`, `collect()`, `sum()`.

In [ ]:
divisible_by_7 = numbers.filter(lambda x: x % 7 == 0)

# Kiểm tra: lấy 5 phần tử đầu để xem
print(divisible_by_7.take(5))

### Bình phương mỗi số

`rdd.map(func)` áp dụng hàm lên **từng phần tử** của RDD, trả về RDD mới cùng kích thước.

Chuỗi `numbers → filter → map` tạo thành lineage graph. Spark tối ưu bằng cách pipeline các transformation lại, không tạo intermediate data.

In [ ]:
squared = divisible_by_7.map(lambda x: x ** 2)

# Xem 5 giá trị đầu sau khi bình phương
print(squared.take(5))

### Tính count, sum, average

Đây là các **action** — trigger toàn bộ lineage graph chạy thực tế:
- `count()` — đếm số phần tử
- `sum()` — tính tổng
- Average = sum / count (không có hàm avg sẵn cho RDD)

In [ ]:
count = squared.count()
total = squared.sum()
average = total / count

print(f"Count: {count}")
print(f"Sum: {total}")
print(f"Average: {average}")

**Đánh giá kết quả:**
- Count = 142 (số từ 1-1000 chia hết cho 7: floor(1000/7) = 142)
- Sum và Average là tổng/bình quân của các số 7², 14², 21², ..., 994²
- Nếu count đúng 142 thì filter hoạt động chính xác ✅

---

## Task 1.2: Text Processing with RDDs

Bài này luyện:
- `flatMap` — giống map nhưng flatten kết quả (list → individual elements)
- `reduceByKey` — gom nhóm theo key và reduce
- `sortBy` — sắp xếp RDD

### Tạo RDD từ text và tách từ

`flatMap` khác `map` ở chỗ:
- `map(lambda line: line.split())` → mỗi phần tử là **một list** `["Apache", "Spark", ...]`
- `flatMap(lambda line: line.split())` → mỗi từ là **một phần tử riêng biệt**

Nói cách khác, flatMap = map + flatten.

In [ ]:
text = [
    "Apache Spark is a unified analytics engine",
    "for large-scale data processing",
    "Spark provides high-level APIs in Java, Scala, Python and R",
    "and supports SQL, streaming, machine learning and graph processing"
]

text_rdd = sc.parallelize(text)
words = text_rdd.flatMap(lambda line: line.split())

print(f"Tổng số từ (chưa làm sạch): {words.count()}")
print(f"10 từ đầu: {words.take(10)}")

### Lowercase và bỏ dấu câu

`map` áp dụng hàm lên từng từ:
- `.lower()` — chuyển về chữ thường
- `.replace(",", "").replace(".", "")` — bỏ dấu phẩy và chấm

Lưu ý: cách này đơn giản nhưng không xử lý hết dấu câu (dấu ngoặc, gạch ngang...). Với bài tập intern level thì đủ.

In [ ]:
clean_words = words.map(lambda word: word.lower().replace(",", "").replace(".", ""))

print(f"10 từ sạch đầu: {clean_words.take(10)}")

### Đếm tần suất từ với reduceByKey

Quy trình word count kinh điển trong MapReduce:
1. `map(lambda w: (w, 1))` — biến mỗi từ thành cặp `(từ, 1)`
2. `reduceByKey(lambda a, b: a + b)` — gom các cặp cùng key, cộng value lại

`reduceByKey` hiệu quả hơn `groupByKey` vì nó reduce **tại chỗ** trước khi shuffle, giảm lượng data truyền giữa các node.

In [ ]:
word_counts = clean_words.map(lambda word: (word, 1)).reduceByKey(lambda a, b: a + b)

print(f"Số từ unique: {word_counts.count()}")
print(f"Một vài cặp: {word_counts.take(5)}")

### Sắp xếp và lấy top 10

`sortBy(keyfunc, ascending=False)` sắp xếp RDD theo giá trị trả về bởi `keyfunc`.

`take(10)` là action — chỉ lấy 10 phần tử đầu, không collect toàn bộ RDD về driver (an toàn với data lớn).

In [ ]:
top_words = word_counts.sortBy(lambda x: x[1], ascending=False).take(10)

print("Top 10 từ xuất hiện nhiều nhất:")
for word, count in top_words:
    print(f"  {word}: {count}")

**Đánh giá kết quả:**
- Các từ như "and", "spark", "processing" nên xuất hiện nhiều lần
- Nếu tất cả count = 1 thì có lỗi ở bước reduceByKey
- Kết quả phải khớp với việc đếm tay trong 4 câu text gốc ✅

---

## Task 1.3: Understanding Partitions

Partition là đơn vị song song cơ bản trong Spark. Mỗi partition được xử lý bởi một task trên một core.

Hiểu partition giúp:
- Tối ưu mức độ song song
- Tránh data skew (một partition quá lớn)
- Giảm shuffle khi join/groupBy

### Kiểm tra số partition mặc định

`getNumPartitions()` cho biết RDD đang có bao nhiêu partition.

Số partition mặc định phụ thuộc vào `spark.default.parallelism` (thường = tổng số core trong cluster).

In [ ]:
data = sc.parallelize(range(1, 101))
print(f"Số partition ban đầu: {data.getNumPartitions()}")

### Repartition thành 4 partitions

`repartition(n)` tạo RDD mới với đúng n partition. Đây là **wide transformation** — gây shuffle toàn bộ data.

Khi nào dùng:
- Data bị skew (một partition quá to)
- Muốn tăng/giảm mức độ song song
- Trước khi join hai RDD có partition key khác nhau

In [ ]:
repartitioned = data.repartition(4)
print(f"Sau repartition: {repartitioned.getNumPartitions()} partitions")

### Xem dữ liệu trong từng partition

`mapPartitions(func)` áp dụng hàm lên **toàn bộ iterator của một partition** thay vì từng phần tử.

Hàm phải nhận iterator và trả về iterator (dùng `yield`). Ở đây ta yield một string mô tả nội dung partition.

In [ ]:
def show_partition_contents(iterator):
    items = list(iterator)
    yield f"Partition contains {len(items)} elements: {items[:5]}..."

partition_info = repartitioned.mapPartitions(show_partition_contents).collect()
for i, info in enumerate(partition_info):
    print(f"Partition {i}: {info}")

**Đánh giá kết quả:**
- Phải có đúng 4 dòng output (4 partitions)
- Tổng số elements across all partitions = 100
- Phân bố nên tương đối đều (~25 elements/partition). Nếu chênh lệch lớn (>2x) thì data bị skew
- Thứ tự số trong mỗi partition có thể không liên tục vì repartition dùng hash partitioner

---

## Tổng kết Exercise 1

| Khái niệm | Transformation (lazy) | Action (trigger) |
|-----------|----------------------|------------------|
| RDD creation | `parallelize` | - |
| Lọc | `filter` | - |
| Biến đổi | `map`, `flatMap` | - |
| Gom nhóm | `reduceByKey` | - |
| Sắp xếp | `sortBy` | - |
| Đếm/tổng | - | `count`, `sum` |
| Lấy dữ liệu | - | `collect`, `take` |
| Partition | `repartition` | `getNumPartitions` |